# 06b: Train MiniT5 Decoder — Two-Phase Training

**Runtime: G4 or A100** — MiniT5 is only ~20M params

**Two-phase approach:**
- **Phase 1:** Copy pretraining on 100K diverse English sentences (AG News)
- **Phase 2:** Fine-tune on structured decoder format (your `decoder_training.jsonl`)

Upload `decoder_training.jsonl` from notebook 06a before running.

**Architecture:** T5-style encoder-decoder, 4 layers, 256-dim, 4 heads, pointer-generator
**Key property:** Zero pretrained knowledge. All facts come from encoder input.

**Time:** ~2-3 hrs on G4 | **Cost:** $0

In [ ]:
!pip install -q sentence-transformers datasets
!nvidia-smi | head -4

In [ ]:
!git clone https://github.com/ibrahimm8x/HCL.git /content/HLC

import sys
sys.path.insert(0, '/content/HLC')

## Upload training data

Upload `decoder_training.jsonl` that you downloaded from notebook 06a.
This is used in **Phase 2** (structured fine-tuning).

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # upload decoder_training.jsonl

DATA_DIR = Path('/content/HLC/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DATA = DATA_DIR / 'decoder_training.jsonl'

# Move uploaded file to data dir
for filename in uploaded:
    Path(filename).rename(TRAINING_DATA)
    print(f'Moved {filename} -> {TRAINING_DATA}')

line_count = sum(1 for _ in open(TRAINING_DATA))
print(f'Training data: {line_count} examples')
print(f'File size: {TRAINING_DATA.stat().st_size / 1024 / 1024:.1f} MB')

In [ ]:
from hlc.config import Config
from hlc.decoder_training import DecoderTrainer

MODEL_DIR = Path('/content/HLC/data/decoder_model')

config = Config(
    data_dir=Path('/content/HLC/data'),
    columns_dir=Path('/content/HLC/data/columns'),
    faiss_dir=Path('/content/HLC/data/faiss_index'),
    decoder_model_path=MODEL_DIR,
)
trainer = DecoderTrainer(config)

print(f'MiniT5 config:')
print(f'  d_model: {config.decoder_d_model}')
print(f'  n_heads: {config.decoder_n_heads}')
print(f'  encoder layers: {config.decoder_n_encoder_layers}')
print(f'  decoder layers: {config.decoder_n_decoder_layers}')
print(f'  FFN dim: {config.decoder_d_ff}')
print(f'  max seq len: {config.decoder_max_seq_len}')

## Phase 1: Copy Pretraining (~1-2 hrs)

Downloads AG News dataset (120K news sentences), trains MiniT5 to copy them verbatim.
This teaches the model English grammar, word order, and vocabulary — zero factual knowledge.

```
Encoder: "The economy grew by 3.5 percent in the third quarter."
Decoder: "<bos> The economy grew by 3.5 percent in the third quarter. <end>"
```

In [ ]:
COPY_DATA = DATA_DIR / 'copy_pretraining.jsonl'

history_phase1 = trainer.pretrain_copy_task(
    output_data_path=COPY_DATA,
    output_model_path=MODEL_DIR,
    num_samples=100000,
    epochs=10,
    batch_size=64,
    learning_rate=1e-4,
    warmup_steps=300,
    max_len=128,
)
print(f'\nPhase 1 complete!')
print(f'Final train loss: {history_phase1["train_loss"][-1]:.4f}')
print(f'Final val loss: {history_phase1["val_loss"][-1]:.4f}')

In [ ]:
# Plot Phase 1 training curves
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(history_phase1['train_loss'], label='Train Loss', marker='o')
ax.plot(history_phase1['val_loss'], label='Val Loss', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Phase 1: Copy Pretraining Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/phase1_loss.png', dpi=100)
plt.show()

## Phase 2: Structured Fine-tuning (~1 hr)

Load Phase 1 checkpoint, fine-tune on the structured decoder format.
Lower learning rate (5e-5) since we have a warm start.

```
Encoder: "<query> What is DNA? <knowledge> DNA contains... <state> joy=0.80..."
Decoder: "<bos> DNA contains the genetic instructions... <end>"
```

In [ ]:
PHASE1_CHECKPOINT = MODEL_DIR / 'model_phase1.pt'

history_phase2 = trainer.train(
    training_data_path=TRAINING_DATA,
    output_model_path=MODEL_DIR,
    epochs=15,
    batch_size=32,
    learning_rate=5e-5,
    warmup_steps=200,
    max_input_len=256,
    max_output_len=64,
    resume_from=PHASE1_CHECKPOINT,
)
print(f'\nPhase 2 complete!')
print(f'Final train loss: {history_phase2["train_loss"][-1]:.4f}')
print(f'Final val loss: {history_phase2["val_loss"][-1]:.4f}')

In [ ]:
# Plot both phases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_phase1['train_loss'], label='Train', marker='o')
axes[0].plot(history_phase1['val_loss'], label='Val', marker='s')
axes[0].set_title('Phase 1: Copy Pretraining')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_phase2['train_loss'], label='Train', marker='o')
axes[1].plot(history_phase2['val_loss'], label='Val', marker='s')
axes[1].set_title('Phase 2: Structured Fine-tuning')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_loss.png', dpi=100)
plt.show()

## Evaluate

Critical tests:
1. **False fact test** — Knowledge says "sun is made of cheese" → must say "cheese"
2. **Empty knowledge test** — No bullets → must say "I don't know"
3. **Faithful composition** — Response only uses provided knowledge
4. **Fluency** — Grammatical, complete English sentences

In [ ]:
from hlc.decoder import Decoder
from hlc.routing import RoutingResult
from hlc.value_system import ValueState
import torch

decoder = Decoder(config)

# === Basic test ===
test_result = RoutingResult(
    converged=True, iterations=1,
    final_state=torch.randn(384),
    active_column_ids=['test-1'],
    active_source_texts=['Water boils at 100 degrees Celsius at standard atmospheric pressure.'],
    prediction_error=0.05,
    value_state=ValueState(joy=0.8, curiosity=0.1),
    mode='fast',
)

response = decoder.generate(test_result, 'What temperature does water boil at?')
print(f'Q: What temperature does water boil at?')
print(f'K: Water boils at 100 degrees Celsius at standard atmospheric pressure.')
print(f'R: {response}')
print()

In [ ]:
# === CRITICAL: False Fact Test ===
# Decoder MUST say "cheese" — proves it has no internal knowledge
false_result = RoutingResult(
    converged=True, iterations=1,
    final_state=torch.randn(384),
    active_column_ids=['false-1'],
    active_source_texts=['The sun is made of cheese.'],
    prediction_error=0.05,
    value_state=ValueState(joy=0.8),
    mode='fast',
)

response = decoder.generate(false_result, 'What is the sun made of?')
print(f'Q: What is the sun made of?')
print(f'K: The sun is made of cheese.')
print(f'R: {response}')
print()
passed = 'cheese' in response.lower()
print(f'FALSE FACT TEST: {"PASS" if passed else "FAIL"}')
if not passed:
    print('WARNING: Decoder may have internal knowledge — this should NOT happen with training from scratch!')

In [ ]:
# === CRITICAL: Empty Knowledge Test ===
# Decoder MUST say "I don't know" — proves no hallucination
empty_result = RoutingResult(
    converged=False, iterations=8,
    final_state=torch.randn(384),
    active_column_ids=[], active_source_texts=[],
    prediction_error=0.9,
    value_state=ValueState(curiosity=0.6, pain=0.2),
    mode='slow',
)

response = decoder.generate(empty_result, 'How do I bake a chocolate cake?')
print(f'Q: How do I bake a chocolate cake?')
print(f'K: (none)')
print(f'R: {response}')
print()
dont_know_phrases = ["don't have", "don't know", "not able", "outside", "no information", "insufficient"]
passed = any(phrase in response.lower() for phrase in dont_know_phrases)
print(f'EMPTY KNOWLEDGE TEST: {"PASS" if passed else "FAIL"}')
if not passed:
    print('WARNING: Decoder may be hallucinating from zero knowledge!')

In [ ]:
# === Batch Evaluation ===
test_cases = [
    ('What is DNA?', ['DNA contains the genetic instructions for the development and function of living organisms.'], 0.8, 0.1),
    ('How fast does light travel?', ['Light travels at approximately 300,000 kilometers per second in a vacuum.'], 0.9, 0.05),
    ('What is Pi?', ['Pi is approximately 3.14159 and represents the ratio of a circle\'s circumference to its diameter.'], 0.7, 0.1),
    ('What are cells?', [
        'All living organisms are made of cells, which are the basic units of life.',
        'Bacteria are single-celled organisms that can be beneficial or harmful to other living things.',
    ], 0.6, 0.2),
    ('What is quantum computing?', [], 0.0, 0.8),
    ('Who won the Super Bowl?', [], 0.0, 0.5),
    ('What is fire?', ['Fire requires fuel, heat, and oxygen to burn. Remove any one and the fire goes out.'], 0.7, 0.0),
]

print('=== Batch Evaluation ===')
for query, knowledge, joy, curiosity in test_cases:
    result = RoutingResult(
        converged=bool(knowledge),
        iterations=1 if knowledge else 8,
        final_state=torch.randn(384),
        active_column_ids=[f'col-{i}' for i in range(len(knowledge))],
        active_source_texts=knowledge,
        prediction_error=0.1 if knowledge else 0.9,
        value_state=ValueState(joy=joy, curiosity=curiosity),
        mode='fast' if joy > 0.7 else 'slow',
    )
    response = decoder.generate(result, query)
    print(f'\nQ: {query}')
    print(f'K: {knowledge[0][:60]}...' if knowledge else 'K: (none)')
    print(f'R: {response}')

## Step 3: Download model (~80 MB)

Save to your local machine at `data/decoder_model/`.

In [ ]:
import shutil

model_path = config.decoder_model_path
if model_path.exists():
    total = sum(f.stat().st_size for f in model_path.rglob('*') if f.is_file())
    print(f'Model at: {model_path}')
    print(f'Total size: {total / 1024 / 1024:.1f} MB')
    for f in sorted(model_path.rglob('*')):
        if f.is_file():
            print(f'  {f.name}: {f.stat().st_size / 1024:.0f} KB')

    # Zip and download
    zip_path = '/content/decoder_model.zip'
    shutil.make_archive('/content/decoder_model', 'zip', str(model_path))
    print(f'\nZipped to: {zip_path}')
    print(f'Zip size: {Path(zip_path).stat().st_size / 1024 / 1024:.1f} MB')

    files.download(zip_path)
else:
    print('Model not found. Training may have failed.')